# WC26 — Predicting the Final 8 Knockout Matches

Using **player behavior** and **squad composition** from the 96 completed matches to predict the 8 remaining knockout fixtures (4 QF, 2 SF, 3rd-place, Final).

**As of 2026-07-14:** Group + R32 + R16 complete (96/104). 8 teams alive.
**Approach:** opponent-adjusted attack/defense ratings (goals blended with xG, regularized toward a squad-strength prior) → Poisson goals model → 10k Monte Carlo bracket simulation.

In [1]:
import dataclaw_data
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 40)
DS = 'b3ee5433'
def load(t): return dataclaw_data.get_dataframe(DS, table_name=t)

teams      = load('file_archive_teams_csv')
matches    = load('file_archive_matches_csv')
stages     = load('file_archive_tournament_stages_csv')
lineups    = load('file_archive_match_lineups_csv')
events     = load('file_archive_match_events_csv')
tstats     = load('file_archive_match_team_stats_csv')
pstats     = load('file_archive_player_stats_csv')
squads     = load('file_archive_squads_and_players_csv')
referees   = load('file_archive_referees_csv')

# cast VARCHAR numeric cols in player_stats
for c in ['shots','shots_on_target','average_rating']:
    pstats[c] = pd.to_numeric(pstats[c], errors='coerce')

TEAM = teams.set_index('team_id')['team_name'].to_dict()
CODE = teams.set_index('team_id')['fifa_code'].to_dict()
ELO  = teams.set_index('team_id')['elo_rating'].to_dict()
RANK = teams.set_index('team_id')['fifa_ranking_pre_tournament'].to_dict()

print('rows:', {k: len(v) for k,v in dict(teams=teams,matches=matches,lineups=lineups,events=events,
      tstats=tstats,pstats=pstats,squads=squads,referees=referees).items()})
print('match status:', matches['status'].value_counts().to_dict())
print('player_stats NaN after cast:', pstats[['shots','shots_on_target','average_rating']].isna().sum().to_dict())

rows: {'teams': 48, 'matches': 104, 'lineups': 4992, 'events': 758, 'tstats': 192, 'pstats': 1248, 'squads': 1248, 'referees': 28}
match status: {'Completed': 96, 'Scheduled': 8}
player_stats NaN after cast: {'shots': 1248, 'shots_on_target': 1248, 'average_rating': 1248}


## 1 · Modeling base & bracket
Long team-match table (2 rows per completed match) + reconstructed 8-match bracket. Knockout home/away is bracket seeding only → treated as neutral.

In [2]:
comp = matches[matches.status=='Completed'].copy()

# long team-match table: one row per team per match
def side(df, home=True):
    p = 'home' if home else 'away'; o = 'away' if home else 'home'
    r = pd.DataFrame({
        'match_id': df.match_id, 'stage_id': df.stage_id,
        'team_id': df[f'{p}_team_id'], 'opp_id': df[f'{o}_team_id'],
        'gf': df[f'{p}_score'], 'ga': df[f'{o}_score'],
        'xg_for': df[f'{p}_xg'], 'xg_ag': df[f'{o}_xg'],
        'is_home': 1 if home else 0,
        'pen_for': df[f'{p}_penalty_score'], 'pen_ag': df[f'{o}_penalty_score'],
    })
    return r
tm = pd.concat([side(comp,True), side(comp,False)], ignore_index=True)
# merge team-match shot stats
tm = tm.merge(tstats[['match_id','team_id','possession_pct','total_shots','shots_on_target','corners','fouls']],
              on=['match_id','team_id'], how='left')
tm['is_knockout'] = tm.stage_id >= 2
print("team-match rows:", len(tm), "| completed matches:", comp.shape[0])
print("home advantage (group stage): home gf %.2f vs away gf %.2f" %
      (tm[(tm.stage_id==1)&(tm.is_home==1)].gf.mean(), tm[(tm.stage_id==1)&(tm.is_home==0)].gf.mean()))

# 8 alive teams (R16 winners) and reconstructed bracket
ALIVE = {33:'France',10:'Morocco',36:'Norway',45:'England',
         None:None}  # placeholder, fill below via query result
# team_ids for alive teams from names
name2id = {v:k for k,v in TEAM.items()}
alive_names = ['Argentina','Spain','France','England','Belgium','Morocco','Switzerland','Norway']
alive = {n: name2id[n] for n in alive_names}
print("\nAlive team_ids:", alive)

# remaining bracket (QF98=ESP-BEL, QF100=ARG-SUI inferred from R16 adjacency)
bracket = {
 'QF97': ('France','Morocco'), 'QF98': ('Spain','Belgium'),
 'QF99': ('Norway','England'), 'QF100': ('Argentina','Switzerland'),
 'SF101': ('QF97','QF98'), 'SF102': ('QF99','QF100'),
 'FINAL': ('SF101','SF102'), 'THIRD': ('SF101L','SF102L'),
}
bracket

team-match rows: 192 | completed matches: 96
home advantage (group stage): home gf 1.79 vs away gf 1.19

Alive team_ids: {'Argentina': 37, 'Spain': 29, 'France': 33, 'England': 45, 'Belgium': 25, 'Morocco': 10, 'Switzerland': 8, 'Norway': 36}


{'QF97': ('France', 'Morocco'),
 'QF98': ('Spain', 'Belgium'),
 'QF99': ('Norway', 'England'),
 'QF100': ('Argentina', 'Switzerland'),
 'SF101': ('QF97', 'QF98'),
 'SF102': ('QF99', 'QF100'),
 'FINAL': ('SF101', 'SF102'),
 'THIRD': ('SF101L', 'SF102L')}

## 2 · Squad composition & player-behavior features
Per-team: top-15 market value, age/experience/height profile, positional depth (composition); goals, assists, goal concentration, discipline (behavior). Combined into a **squad-strength prior** used to regularize the on-pitch ratings.

In [3]:
REF = pd.Timestamp('2026-07-14')
sq = squads.copy()
sq['dob'] = pd.to_datetime(sq['date_of_birth'], errors='coerce')
sq['age'] = (REF - sq['dob']).dt.days/365.25
def posgroup(p):
    p=str(p).upper()
    if 'GK' in p or 'KEEP' in p: return 'GK'
    if any(k in p for k in ['CB','LB','RB','DEF','BACK','WB']): return 'DEF'
    if any(k in p for k in ['DM','CM','AM','MID','MF']): return 'MID'
    return 'FWD'
sq['posg'] = sq['position'].map(posgroup)

def squad_feats(g):
    g = g.sort_values('market_value_eur', ascending=False)
    top15 = g.head(15)
    depth = g['posg'].value_counts()
    return pd.Series({
        'squad_val_top15': top15['market_value_eur'].sum(),
        'squad_val_total': g['market_value_eur'].sum(),
        'avg_age': g['age'].mean(),
        'avg_caps': g['caps'].mean(),
        'star_caps': top15['caps'].mean(),
        'avg_height': g['height_cm'].mean(),
        'n_players': len(g),
        'depth_DEF': depth.get('DEF',0), 'depth_MID': depth.get('MID',0), 'depth_FWD': depth.get('FWD',0),
    })
scomp = sq.groupby('team_id').apply(squad_feats, include_groups=False)

# player behavior from player_stats (team-level)
def behav(g):
    tg = g['goals'].sum()
    top = g.nlargest(1,'goals')
    return pd.Series({
        'goals_total': tg,
        'assists_total': g['assists'].sum(),
        'top_scorer': top['player_name'].iloc[0] if len(top) else None,
        'top_scorer_goals': int(top['goals'].iloc[0]) if len(top) else 0,
        'top1_share': (top['goals'].iloc[0]/tg) if tg>0 else 0,      # reliance on one striker
        'yellows': g['yellow_cards'].sum(), 'reds': g['red_cards'].sum(),
        'n_scorers': int((g['goals']>0).sum()),
    })
pbeh = pstats.groupby('team_id').apply(behav, include_groups=False)

# matches played per team (for shrinkage) + goals/xg aggregates
mp = tm.groupby('team_id').agg(mp=('match_id','count'),
        gf=('gf','sum'), ga=('ga','sum'), xgf=('xg_for','sum'), xga=('xg_ag','sum')).reset_index()

feat = (mp.merge(scomp.reset_index(), on='team_id', how='left')
          .merge(pbeh.reset_index(), on='team_id', how='left'))
feat['team'] = feat.team_id.map(TEAM); feat['elo']=feat.team_id.map(ELO); feat['rank']=feat.team_id.map(RANK)
feat['disc_per_match'] = (feat.yellows + 3*feat.reds)/feat.mp

# strength prior: z-score elo, log top15 value, star caps  -> composite
import numpy as np
def z(s): return (s-s.mean())/s.std(ddof=0)
feat['prior_raw'] = 0.55*z(feat.elo) + 0.30*z(np.log(feat.squad_val_top15)) + 0.15*z(feat.star_caps)
feat['prior_z'] = z(feat.prior_raw)

cols=['team','elo','rank','squad_val_top15','avg_age','star_caps','goals_total','top_scorer','top_scorer_goals','top1_share','disc_per_match','prior_z']
print(feat[feat.team.isin(alive_names)].sort_values('prior_z',ascending=False)[cols].round(2).to_string(index=False))

       team  elo  rank  squad_val_top15  avg_age  star_caps  goals_total           top_scorer  top_scorer_goals  top1_share  disc_per_match  prior_z
      Spain 2120     2     1572399628.0    26.84      31.67            8      Mikel Oyarzabal                 4        0.50             0.6     2.02
  Argentina 2150     1      937539444.0    29.21      33.13           13  Lionel Andrés Messi                 8        0.62             0.4     1.98
     France 2100     3     1313204469.0    27.09      30.27           14        Kylian Mbappe                 7        0.50             0.2     1.85
    England 2050     4     1102913351.0    27.36      28.53           11    Harry Edward Kane                 6        0.55             2.0     1.54
    Belgium 1950     9      485117492.0    27.70      29.40           12        Romelu Lukaku                 3        0.25             1.4     0.84
    Morocco 1920     7      413054154.0    26.69      28.00           10       Ismael Saibari             

## 3 · Attack/defense ratings + holdout validation
Iterative opponent-adjusted attack/defense factors on **blended goals** (0.6·xG + 0.4·actual, xG is more stable), shrunk toward the squad-strength prior (weight = mp/(mp+5)). **Validation:** fit on group stage only, predict the 24 completed knockouts.

In [4]:
from scipy.stats import poisson
PRIORZ = feat.set_index('team_id')['prior_z'].to_dict()
ALL_TEAMS = feat.team_id.tolist()

def fit_ratings(tm_fit, w_xg=0.6, K=5.0, beta=0.16, n_iter=200):
    d = tm_fit.copy()
    d['gs_for'] = w_xg*d.xg_for + (1-w_xg)*d.gf
    d['gs_ag']  = w_xg*d.xg_ag + (1-w_xg)*d.ga
    mu = d['gs_for'].mean()
    mpd = d.groupby('team_id').size().to_dict()
    atk = {t:1.0 for t in ALL_TEAMS}; dfn = {t:1.0 for t in ALL_TEAMS}; gamma=1.3
    rows = list(d.itertuples())
    for _ in range(n_iter):
        na={t:0. for t in ALL_TEAMS}; da={t:1e-9 for t in ALL_TEAMS}
        nd={t:0. for t in ALL_TEAMS}; dd={t:1e-9 for t in ALL_TEAMS}
        ghn=0.; ghd=1e-9
        for r in rows:
            i,j,h = r.team_id, r.opp_id, r.is_home
            na[i]+=r.gs_for;  da[i]+=mu*dfn[j]*(gamma if h else 1)
            nd[i]+=r.gs_ag;   dd[i]+=mu*atk[j]*(gamma if not h else 1)
            if h: ghn+=r.gs_for; ghd+=mu*atk[i]*dfn[j]
        for t in ALL_TEAMS:
            if t in mpd: atk[t]=na[t]/da[t]; dfn[t]=nd[t]/dd[t]
        ma=np.mean([atk[t] for t in ALL_TEAMS]); md=np.mean([dfn[t] for t in ALL_TEAMS])
        for t in ALL_TEAMS: atk[t]/=ma; dfn[t]/=md
        gamma=ghn/ghd
    # shrink toward prior
    af={}; df_={}
    for t in ALL_TEAMS:
        n=mpd.get(t,0); w=n/(n+K)
        af[t]=np.exp(w*np.log(atk[t]) + (1-w)*( beta*PRIORZ[t]))
        df_[t]=np.exp(w*np.log(dfn[t]) + (1-w)*(-beta*PRIORZ[t]))
    return dict(mu=mu, atk=af, dfn=df_, gamma=gamma)

def match_probs(lamA,lamB,mx=12):
    pa=poisson.pmf(np.arange(mx+1),lamA); pb=poisson.pmf(np.arange(mx+1),lamB)
    M=np.outer(pa,pb)
    return np.tril(M,-1).sum(), np.trace(M), np.triu(M,1).sum()   # pA, draw, pB

def strength(R,t): return np.log(R['atk'][t]) - np.log(R['dfn'][t])
def pen_p(R,A,B): return 1/(1+np.exp(-0.4*(strength(R,A)-strength(R,B))))   # mild favorite edge

def adv_prob(R,A,B):   # prob A advances in a knockout (neutral)
    lamA=R['mu']*R['atk'][A]*R['dfn'][B]; lamB=R['mu']*R['atk'][B]*R['dfn'][A]
    pA,pd,pB=match_probs(lamA,lamB)
    return pA + pd*pen_p(R,A,B), (lamA,lamB,pA,pd,pB)

# ---- VALIDATION: fit on group stage, predict completed knockouts ----
Rg = fit_ratings(tm[tm.stage_id==1])
ko = comp[comp.stage_id>=2].copy()
def home_advanced(r):
    if r.home_score!=r.away_score: return int(r.home_score>r.away_score)
    return int(r.home_penalty_score>r.away_penalty_score)
y, p = [], []
for r in ko.itertuples():
    p_home,_ = adv_prob(Rg, r.home_team_id, r.away_team_id)
    p.append(p_home); y.append(home_advanced(r))
y=np.array(y); p=np.clip(np.array(p),1e-6,1-1e-6)
acc=((p>0.5)==(y==1)).mean(); ll=-(y*np.log(p)+(1-y)*np.log(1-p)).mean(); brier=((p-y)**2).mean()
base=-(y.mean()*np.log(y.mean())+(1-y.mean())*np.log(1-y.mean()))
print(f"Holdout on {len(y)} completed knockouts (group-only ratings):")
print(f"  accuracy={acc:.3f}   log-loss={ll:.3f} (baseline {base:.3f})   brier={brier:.3f}")
# calibration in 3 buckets
dfc=pd.DataFrame({'p':p,'y':y}); dfc['bk']=pd.cut(dfc.p,[0,.4,.6,1])
print(dfc.groupby('bk',observed=True).agg(pred=('p','mean'),actual=('y','mean'),n=('y','size')).round(2).to_string())

Holdout on 24 completed knockouts (group-only ratings):
  accuracy=0.667   log-loss=0.544 (baseline 0.690)   brier=0.181
            pred  actual   n
bk                          
(0.0, 0.4]  0.31    0.00   3
(0.4, 0.6]  0.53    0.36  11
(0.6, 1.0]  0.73    0.90  10


## 4 · Final ratings, QF predictions & 10k Monte Carlo
Refit on all 96 completed matches. All four QF matchups are now known → direct analytic predictions. SF/Final/3rd depend on results → 10k bracket simulations (extra time + penalty shootouts) for advancement and title odds.

In [5]:
R = fit_ratings(tm)   # final ratings on all 96 completed matches

# alive team rating table
rt = pd.DataFrame({'team_id':list(alive.values())})
rt['team']=rt.team_id.map(TEAM)
rt['atk']=rt.team_id.map(R['atk']); rt['dfn']=rt.team_id.map(R['dfn'])
rt['strength']=rt.team_id.apply(lambda t: strength(R,t))
rt['xg_for_vs_avg']=R['mu']*rt.atk; rt['xg_ag_vs_avg']=R['mu']*rt.dfn
rt=rt.sort_values('strength',ascending=False).reset_index(drop=True)
print("FINAL TEAM RATINGS (alive 8), sorted by overall strength:")
print(rt[['team','atk','dfn','strength','xg_for_vs_avg','xg_ag_vs_avg']].round(3).to_string(index=False))

def scoreline(lamA,lamB,mx=8):
    pa=poisson.pmf(np.arange(mx+1),lamA); pb=poisson.pmf(np.arange(mx+1),lamB)
    M=np.outer(pa,pb); i,j=np.unravel_index(M.argmax(),M.shape); return i,j,M.max()

QF=[('QF97','France','Morocco'),('QF98','Spain','Belgium'),('QF99','Norway','England'),('QF100','Argentina','Switzerland')]
print("\nQUARTER-FINAL DIRECT PREDICTIONS (neutral venue):")
qf_rows=[]
for tag,A,B in QF:
    a,b=alive[A],alive[B]
    lamA=R['mu']*R['atk'][a]*R['dfn'][b]; lamB=R['mu']*R['atk'][b]*R['dfn'][a]
    pA,pd_,pB=match_probs(lamA,lamB)
    advA=pA+pd_*pen_p(R,a,b)
    si,sj,_=scoreline(lamA,lamB)
    qf_rows.append(dict(match=tag,A=A,B=B,padvA=advA,padvB=1-advA,lamA=lamA,lamB=lamB,ml=f"{si}-{sj}"))
    print(f"  {A} vs {B}: {A} {advA*100:4.1f}% | {B} {(1-advA)*100:4.1f}%  (xG {lamA:.2f}-{lamB:.2f}, likely {si}-{sj})")
qf_df=pd.DataFrame(qf_rows)

# ---- Monte Carlo bracket ----
rng=np.random.default_rng(7)
def sim_ko(A,B):
    lamA=R['mu']*R['atk'][A]*R['dfn'][B]; lamB=R['mu']*R['atk'][B]*R['dfn'][A]
    gA=rng.poisson(lamA); gB=rng.poisson(lamB)
    if gA==gB:
        gA+=rng.poisson(lamA/3); gB+=rng.poisson(lamB/3)   # extra time
        if gA==gB:
            return (A if rng.random()<pen_p(R,A,B) else B)
    return A if gA>gB else B
def other(pair,w): return pair[0] if w==pair[1] else pair[1]

N=10000
from collections import Counter
champ=Counter(); finalist=Counter(); sf=Counter(); third=Counter(); finals=Counter()
for _ in range(N):
    w97=sim_ko(alive['France'],alive['Morocco']); w98=sim_ko(alive['Spain'],alive['Belgium'])
    w99=sim_ko(alive['Norway'],alive['England']); w100=sim_ko(alive['Argentina'],alive['Switzerland'])
    for t in (w97,w98,w99,w100): sf[t]+=1
    s1=(w97,w98); s2=(w99,w100)
    f1=sim_ko(*s1); f2=sim_ko(*s2)
    finalist[f1]+=1; finalist[f2]+=1
    l1=other(s1,f1); l2=other(s2,f2)
    ch=sim_ko(f1,f2); champ[ch]+=1
    finals[tuple(sorted((TEAM[f1],TEAM[f2])))]+=1
    third[sim_ko(l1,l2)]+=1

ids={v:k for k,v in alive.items()}
proj=pd.DataFrame({'team':list(alive.keys())})
proj['team_id']=proj.team.map(alive)
proj['reach_SF']=proj.team_id.map(lambda t: sf[t]/N)
proj['reach_Final']=proj.team_id.map(lambda t: finalist[t]/N)
proj['champion']=proj.team_id.map(lambda t: champ[t]/N)
proj['third_place']=proj.team_id.map(lambda t: third[t]/N)
proj['podium']=proj.champion+proj.team_id.map(lambda t: finalist[t]/N - champ[t]/N)+proj.third_place
proj=proj.sort_values('champion',ascending=False).reset_index(drop=True)
print("\nMONTE-CARLO TOURNAMENT PROJECTION (10k sims):")
print(proj[['team','reach_SF','reach_Final','champion','third_place']].round(3).to_string(index=False))
print("\nMost likely FINAL matchups:")
for m,c in finals.most_common(5): print(f"  {m[0]} vs {m[1]}: {c/N*100:.1f}%")

FINAL TEAM RATINGS (alive 8), sorted by overall strength:
       team   atk   dfn  strength  xg_for_vs_avg  xg_ag_vs_avg
      Spain 1.394 0.404     1.239          1.926         0.558
     France 1.586 0.481     1.194          2.191         0.664
    Morocco 1.407 0.624     0.814          1.944         0.861
     Norway 1.532 0.838     0.603          2.117         1.158
  Argentina 1.358 0.743     0.603          1.876         1.026
    Belgium 1.329 0.731     0.597          1.836         1.010
    England 1.384 0.913     0.416          1.913         1.261
Switzerland 1.065 0.843     0.234          1.471         1.165

QUARTER-FINAL DIRECT PREDICTIONS (neutral venue):
  France vs Morocco: France 61.6% | Morocco 38.4%  (xG 1.37-0.93, likely 1-0)
  Spain vs Belgium: Spain 68.4% | Belgium 31.6%  (xG 1.41-0.74, likely 1-0)
  Norway vs England: Norway 57.1% | England 42.9%  (xG 1.93-1.60, likely 1-1)
  Argentina vs Switzerland: Argentina 62.1% | Switzerland 37.9%  (xG 1.58-1.09, likely 1-1)


## 5 · Visualizations

In [6]:
import os, matplotlib as mpl
mpl.rcParams.update({'font.size':11,'axes.grid':True,'grid.alpha':.25,'axes.spines.top':False,'axes.spines.right':False,'figure.dpi':130})
CWD=os.getcwd(); print("saving to", CWD)
COL={'Spain':'#E4572E','France':'#2E4A8B','Argentina':'#4FA9E0','Norway':'#7B2D40',
     'Morocco':'#148F55','England':'#C0392B','Belgium':'#E1A700','Switzerland':'#7D3C98'}
paths={}

# --- FIG 1: Title race (champion % + reach-final %) ---
d=proj.sort_values('champion')
fig,ax=plt.subplots(figsize=(8.6,5))
ax.barh(d.team, d.reach_Final*100, color='#dfe3ea', label='Reach Final', zorder=1)
ax.barh(d.team, d.champion*100, color=[COL[t] for t in d.team], label='Win title', zorder=2)
for y,(c,f) in enumerate(zip(d.champion,d.reach_Final)):
    ax.text(c*100+0.4, y, f"{c*100:.0f}%", va='center', fontweight='bold', fontsize=10)
    ax.text(f*100+0.4, y, f"{f*100:.0f}%", va='center', color='#7a8190', fontsize=8.5)
ax.set_xlabel('Probability (%)'); ax.set_title('WC26 Title Race — champion vs reach-final odds (10k sims)',fontweight='bold')
ax.legend(loc='lower right',frameon=False); fig.tight_layout()
p=f'{CWD}/wc26_f8_title_race.png'; fig.savefig(p,bbox_inches='tight'); plt.close(fig); paths['title']=p

# --- FIG 2: Strength map (attack vs defense) ---
fig,ax=plt.subplots(figsize=(8.6,6))
for _,r in rt.iterrows():
    x,yv=r.xg_for_vs_avg, r.xg_ag_vs_avg
    ax.scatter(x,yv,s=300+proj.set_index('team').champion[r.team]*4500,color=COL[r.team],alpha=.85,edgecolor='white',lw=1.5,zorder=3)
    ax.annotate(r.team,(x,yv),xytext=(0,-4),textcoords='offset points',ha='center',va='top',fontsize=9,fontweight='bold')
ax.axvline(R['mu'],color='#aaa',ls='--',lw=.8); ax.axhline(R['mu'],color='#aaa',ls='--',lw=.8)
ax.invert_yaxis()  # up = fewer goals conceded = better defense
ax.set_xlabel('Attack →  expected goals for (per match)'); ax.set_ylabel('← Defense  expected goals against')
ax.set_title('Team Strength Map — bubble size = title odds\n(top-right = elite both ends)',fontweight='bold')
ax.text(.99,.02,'dashed = tournament average',transform=ax.transAxes,ha='right',color='#999',fontsize=8)
fig.tight_layout(); p=f'{CWD}/wc26_f8_strength_map.png'; fig.savefig(p,bbox_inches='tight'); plt.close(fig); paths['map']=p

# --- FIG 3: Quarter-final win probabilities ---
fig,ax=plt.subplots(figsize=(8.6,4.6))
qd=qf_df.iloc[::-1].reset_index(drop=True)
for y,r in qd.iterrows():
    ax.barh(y, -r.padvA*100, color=COL[r.A]); ax.barh(y, r.padvB*100, color=COL[r.B])
    ax.text(-r.padvA*100-1, y, f"{r.A} {r.padvA*100:.0f}%", va='center', ha='right', fontsize=9.5, fontweight='bold')
    ax.text(r.padvB*100+1, y, f"{r.B} {r.padvB*100:.0f}%", va='center', ha='left', fontsize=9.5, fontweight='bold')
    ax.text(0,y+0.34,f"xG {r.lamA:.2f}–{r.lamB:.2f} · likely {r.ml}",va='bottom',ha='center',fontsize=7.8,color='#666')
ax.axvline(0,color='#333',lw=1); ax.set_xlim(-100,100); ax.set_yticks([]); ax.set_xlabel('← advance %      advance % →')
ax.set_title('Quarter-final Predictions (all 4 matchups known)',fontweight='bold'); ax.grid(False)
fig.tight_layout(); p=f'{CWD}/wc26_f8_qf.png'; fig.savefig(p,bbox_inches='tight'); plt.close(fig); paths['qf']=p

# --- FIG 4: Advancement funnel ---
fig,ax=plt.subplots(figsize=(8.6,5))
d=proj.sort_values('champion',ascending=False); x=np.arange(len(d)); w=.26
ax.bar(x-w, d.reach_SF*100, w, label='Reach SF', color='#b9c2d0')
ax.bar(x,   d.reach_Final*100, w, label='Reach Final', color='#6f7f99')
ax.bar(x+w, d.champion*100, w, label='Champion', color=[COL[t] for t in d.team])
ax.set_xticks(x); ax.set_xticklabels(d.team,rotation=25,ha='right'); ax.set_ylabel('Probability (%)')
ax.set_title('Advancement Funnel by Team',fontweight='bold'); ax.legend(frameon=False)
fig.tight_layout(); p=f'{CWD}/wc26_f8_funnel.png'; fig.savefig(p,bbox_inches='tight'); plt.close(fig); paths['funnel']=p

# --- FIG 5: Player behavior & squad composition ---
fb=feat[feat.team.isin(alive_names)].copy().sort_values('goals_total')
fig,(a1,a2)=plt.subplots(1,2,figsize=(11.5,5))
a1.barh(fb.team, fb.goals_total, color=[COL[t] for t in fb.team])
for y,(g,ts,tg,sh) in enumerate(zip(fb.goals_total,fb.top_scorer,fb.top_scorer_goals,fb.top1_share)):
    a1.text(g+0.2,y,f"{str(ts).split()[-1]} {tg} ({sh*100:.0f}%)",va='center',fontsize=8.2,color='#444')
a1.set_title('Goals scored & top-scorer reliance',fontweight='bold'); a1.set_xlabel('Team goals (tournament)')
a1.set_xlim(0,fb.goals_total.max()+6)
sc=a2.scatter(fb.squad_val_top15/1e9, fb.disc_per_match, s=180, color=[COL[t] for t in fb.team], edgecolor='white', lw=1.4)
for _,r in fb.iterrows(): a2.annotate(r.team,(r.squad_val_top15/1e9,r.disc_per_match),xytext=(5,3),textcoords='offset points',fontsize=8.5,fontweight='bold')
a2.set_xlabel('Top-15 squad market value (€B)'); a2.set_ylabel('Discipline risk (cards/match, red=×3)')
a2.set_title('Squad value vs discipline risk',fontweight='bold')
fig.suptitle('Player Behavior & Squad Composition — the 8 live teams',fontweight='bold',y=1.02)
fig.tight_layout(); p=f'{CWD}/wc26_f8_players.png'; fig.savefig(p,bbox_inches='tight'); plt.close(fig); paths['players']=p

print("saved:", list(paths.values()))

saving to /Users/nandinimathan/.dataclaw/workspaces/e3699c09-6bb4-4856-a880-fb887939e50b


saved: ['/Users/nandinimathan/.dataclaw/workspaces/e3699c09-6bb4-4856-a880-fb887939e50b/wc26_f8_title_race.png', '/Users/nandinimathan/.dataclaw/workspaces/e3699c09-6bb4-4856-a880-fb887939e50b/wc26_f8_strength_map.png', '/Users/nandinimathan/.dataclaw/workspaces/e3699c09-6bb4-4856-a880-fb887939e50b/wc26_f8_qf.png', '/Users/nandinimathan/.dataclaw/workspaces/e3699c09-6bb4-4856-a880-fb887939e50b/wc26_f8_funnel.png', '/Users/nandinimathan/.dataclaw/workspaces/e3699c09-6bb4-4856-a880-fb887939e50b/wc26_f8_players.png']


In [7]:
import plotly.graph_objects as go, json
FONT=dict(family="Inter, Segoe UI, sans-serif")
def savef(fig, name):
    fig.update_layout(font=FONT, template='plotly_white', margin=dict(l=60,r=30,t=60,b=50))
    path=f'{CWD}/{name}.json'
    open(path,'w').write(fig.to_json())
    return path
figpaths={}
order=proj.sort_values('champion')

# 1. Title race
f=go.Figure()
f.add_bar(y=order.team, x=(order.reach_Final*100).round(1), orientation='h', name='Reach Final',
          marker_color='#dfe3ea', hovertemplate='%{y}: %{x}% reach final<extra></extra>')
f.add_bar(y=order.team, x=(order.champion*100).round(1), orientation='h', name='Win title',
          marker_color=[COL[t] for t in order.team], text=[f"{c*100:.0f}%" for c in order.champion],
          textposition='outside', hovertemplate='%{y}: %{x}% champion<extra></extra>')
f.update_layout(barmode='overlay', title='Title Race — champion vs reach-final odds (10k sims)',
                xaxis_title='Probability (%)', legend=dict(orientation='h',y=-0.15))
figpaths['title']=savef(f,'fig_title')

# 2. Quarter-finals diverging
f=go.Figure()
qd=qf_df.iloc[::-1]
f.add_bar(y=qd.A, x=-(qd.padvA*100).round(1), orientation='h', marker_color=[COL[t] for t in qd.A],
          text=[f"{a} {p*100:.0f}%" for a,p in zip(qd.A,qd.padvA)], textposition='outside',
          hovertemplate='%{text}<extra></extra>', showlegend=False)
f.add_bar(y=qd.B, x=(qd.padvB*100).round(1), orientation='h', marker_color=[COL[t] for t in qd.B],
          text=[f"{b} {p*100:.0f}%" for b,p in zip(qd.B,qd.padvB)], textposition='outside',
          hovertemplate='%{text}<extra></extra>', showlegend=False)
f.update_layout(barmode='relative', title='Quarter-final Predictions — advance probability (all 4 matchups known)',
                xaxis=dict(title='← advance %      advance % →', range=[-115,115]),
                yaxis=dict(showticklabels=False))
for i,(_,r) in enumerate(qd.iterrows()):
    f.add_annotation(x=0, y=r.A, yshift=15, text=f"xG {r.lamA:.2f}–{r.lamB:.2f} · likely {r.ml}",
                     showarrow=False, font=dict(size=10,color='#777'))
figpaths['qf']=savef(f,'fig_qf')

# 3. Advancement funnel
d=proj.sort_values('champion',ascending=False)
f=go.Figure()
f.add_bar(x=d.team, y=(d.reach_SF*100).round(1), name='Reach SF', marker_color='#b9c2d0')
f.add_bar(x=d.team, y=(d.reach_Final*100).round(1), name='Reach Final', marker_color='#6f7f99')
f.add_bar(x=d.team, y=(d.champion*100).round(1), name='Champion', marker_color=[COL[t] for t in d.team])
f.update_layout(barmode='group', title='Advancement Funnel by Team', yaxis_title='Probability (%)',
                legend=dict(orientation='h',y=-0.2))
figpaths['funnel']=savef(f,'fig_funnel')

# 4. Strength map
f=go.Figure()
f.add_scatter(x=rt.xg_for_vs_avg, y=rt.xg_ag_vs_avg, mode='markers+text',
              text=rt.team, textposition='bottom center',
              marker=dict(size=[18+proj.set_index('team').champion[t]*160 for t in rt.team],
                          color=[COL[t] for t in rt.team], line=dict(color='white',width=1.5)),
              hovertemplate='%{text}<br>xG for %{x:.2f} · xG against %{y:.2f}<extra></extra>')
f.add_vline(x=R['mu'], line=dict(color='#bbb',dash='dash')); f.add_hline(y=R['mu'], line=dict(color='#bbb',dash='dash'))
f.update_layout(title='Team Strength Map — bubble = title odds (top-right = elite both ends)',
                xaxis_title='Attack →  expected goals for', yaxis=dict(title='← Defense  expected goals against', autorange='reversed'))
figpaths['map']=savef(f,'fig_map')

print(json.dumps({k:os.path.getsize(v) for k,v in figpaths.items()}, indent=0))
print(list(figpaths.values()))

{
"title": 7858,
"qf": 8306,
"funnel": 7813,
"map": 8126
}
['/Users/nandinimathan/.dataclaw/workspaces/e3699c09-6bb4-4856-a880-fb887939e50b/fig_title.json', '/Users/nandinimathan/.dataclaw/workspaces/e3699c09-6bb4-4856-a880-fb887939e50b/fig_qf.json', '/Users/nandinimathan/.dataclaw/workspaces/e3699c09-6bb4-4856-a880-fb887939e50b/fig_funnel.json', '/Users/nandinimathan/.dataclaw/workspaces/e3699c09-6bb4-4856-a880-fb887939e50b/fig_map.json']


# v2 upgrades
Rigor, uncertainty, realism, sensitivity and better visuals.

## v2.1 · Ablation — do player/squad features beat a plain Elo baseline?
Fit on the group stage, predict the 24 completed knockouts. M0 Elo-only → M1 on-pitch ratings (no prior) → M2 +Elo prior → M3 +squad (value+caps) prior. Isolates what each layer adds.

In [8]:
TIDS=list(feat.team_id); NT=len(TIDS); TI={t:i for i,t in enumerate(TIDS)}
def zc(s): return (s-s.mean())/s.std(ddof=0)
priorz_full = feat.set_index('team_id').reindex(TIDS)['prior_z'].values
priorz_elo  = zc(feat.set_index('team_id').reindex(TIDS)['elo']).values

def fit_fast(sub, weights=None, prior_arr=None, K=5.0, beta=0.16, n_iter=200):
    gs_for=(0.6*sub.xg_for+0.4*sub.gf).values; gs_ag=(0.6*sub.xg_ag+0.4*sub.ga).values
    i=sub.team_id.map(TI).values; j=sub.opp_id.map(TI).values; home=sub.is_home.values.astype(float)
    w=np.ones(len(sub)) if weights is None else np.asarray(weights,float)
    mu=np.average(gs_for,weights=w)
    atk=np.ones(NT); dfn=np.ones(NT); gamma=1.3
    present=np.bincount(i,minlength=NT)>0
    for _ in range(n_iter):
        ha=np.where(home==1,gamma,1.0); hopp=np.where(home==0,gamma,1.0)
        num_a=np.bincount(i,w*gs_for,NT); den_a=np.bincount(i,w*mu*dfn[j]*ha,NT)
        num_d=np.bincount(i,w*gs_ag,NT);  den_d=np.bincount(i,w*mu*atk[j]*hopp,NT)
        with np.errstate(invalid='ignore',divide='ignore'):
            atk=np.where(present&(den_a>0),num_a/np.where(den_a>0,den_a,1),atk)
            dfn=np.where(present&(den_d>0),num_d/np.where(den_d>0,den_d,1),dfn)
        atk[present]/=atk[present].mean(); dfn[present]/=dfn[present].mean()
        hm=home==1
        gamma=(w[hm]*gs_for[hm]).sum()/max((w[hm]*mu*atk[i[hm]]*dfn[j[hm]]).sum(),1e-9)
    mp=np.bincount(i,w,NT)
    if prior_arr is not None:
        sw=mp/(mp+K)
        atk=np.exp(sw*np.log(np.clip(atk,1e-6,None))+(1-sw)*( beta*np.nan_to_num(prior_arr)))
        dfn=np.exp(sw*np.log(np.clip(dfn,1e-6,None))+(1-sw)*(-beta*np.nan_to_num(prior_arr)))
    return dict(mu=mu,atk=atk,dfn=dfn,gamma=gamma)

def mprobs(lamA,lamB,mx=12,rho=0.0):
    a=poisson.pmf(np.arange(mx+1),lamA); b=poisson.pmf(np.arange(mx+1),lamB); M=np.outer(a,b)
    if rho:  # Dixon-Coles low-score correction
        t=np.ones((mx+1,mx+1))
        t[0,0]=1-lamA*lamB*rho; t[0,1]=1+lamA*rho; t[1,0]=1+lamB*rho; t[1,1]=1-rho
        M=M*t; M/=M.sum()
    return np.tril(M,-1).sum(), np.trace(M), np.triu(M,1).sum()
def strn(R,i): return np.log(R['atk'][i])-np.log(R['dfn'][i])
def advp(R,ai,bi,rho=0.0):
    lamA=R['mu']*R['atk'][ai]*R['dfn'][bi]; lamB=R['mu']*R['atk'][bi]*R['dfn'][ai]
    pA,pd,pB=mprobs(lamA,lamB,rho=rho); pen=1/(1+np.exp(-0.4*(strn(R,ai)-strn(R,bi))))
    return pA+pd*pen
def elo_adv(ea,eb): return 1/(1+10**(-(ea-eb)/400))

ko=comp[comp.stage_id>=2].copy()
def home_adv(r): return int(r.home_score>r.away_score) if r.home_score!=r.away_score else int(r.home_penalty_score>r.away_penalty_score)
Y=np.array([home_adv(r) for r in ko.itertuples()])
def metrics(P):
    P=np.clip(P,1e-6,1-1e-6)
    acc=((P>0.5)==(Y==1)).mean(); ll=-(Y*np.log(P)+(1-Y)*np.log(1-P)).mean(); br=((P-Y)**2).mean()
    return acc,ll,br

grp=tm[tm.stage_id==1]
RM1=fit_fast(grp,prior_arr=None); RM2=fit_fast(grp,prior_arr=priorz_elo); RM3=fit_fast(grp,prior_arr=priorz_full)
rows=[]
P0=np.array([elo_adv(ELO[r.home_team_id],ELO[r.away_team_id]) for r in ko.itertuples()]); rows.append(("M0 Elo-only",*metrics(P0)))
for name,R_ in [("M1 on-pitch, no prior",RM1),("M2 +Elo prior",RM2),("M3 +squad prior (full)",RM3)]:
    P=np.array([advp(R_,TI[r.home_team_id],TI[r.away_team_id]) for r in ko.itertuples()]); rows.append((name,*metrics(P)))
abl=pd.DataFrame(rows,columns=["model","accuracy","log_loss","brier"])
print("ABLATION on 24 completed knockouts (fit on group stage):"); print(abl.round(3).to_string(index=False))

ABLATION on 24 completed knockouts (fit on group stage):
                 model  accuracy  log_loss  brier
           M0 Elo-only     0.750     0.516  0.172
 M1 on-pitch, no prior     0.750     0.578  0.194
         M2 +Elo prior     0.792     0.549  0.182
M3 +squad prior (full)     0.750     0.543  0.179


## v2.2 · Model realism — Dixon-Coles, recency weighting, suspensions
(1) Estimate a global Dixon-Coles low-score correlation `rho` by MLE on completed scorelines. (2) Recency-weight matches (half-life 25 days) so knockouts outweigh group openers. (3) Red card in the R16 → suspended for the QF (Quansah/England). Re-validate to check DC helps calibration.

In [9]:
# --- recency weights (half-life 25d) ---
mdate = matches.set_index('match_id')['date']
tm2 = tm.copy(); tm2['date']=pd.to_datetime(tm2.match_id.map(mdate)); 
tm2['days_ago']=(REF - tm2['date']).dt.days
HALF=25.0; tm2['w']=0.5**(tm2['days_ago']/HALF)

# --- estimate Dixon-Coles rho by MLE on completed scorelines (using full-96 ratings) ---
Rprod0 = fit_fast(tm2, weights=tm2['w'].values, prior_arr=priorz_full)   # recency-weighted, full prior
def dc_tau(gA,gB,lamA,lamB,rho):
    if gA==0 and gB==0: return 1-lamA*lamB*rho
    if gA==0 and gB==1: return 1+lamA*rho
    if gA==1 and gB==0: return 1+lamB*rho
    if gA==1 and gB==1: return 1-rho
    return 1.0
def loglik(rho):
    ll=0.0
    for r in comp.itertuples():
        ai,bi=TI[r.home_team_id],TI[r.away_team_id]
        lamA=Rprod0['mu']*Rprod0['atk'][ai]*Rprod0['dfn'][bi]*Rprod0['gamma']  # home gets gamma in-sample
        lamB=Rprod0['mu']*Rprod0['atk'][bi]*Rprod0['dfn'][ai]
        pa=poisson.pmf(r.home_score,lamA); pb=poisson.pmf(r.away_score,lamB)
        tau=dc_tau(r.home_score,r.away_score,lamA,lamB,rho)
        ll+=np.log(max(pa*pb*tau,1e-12))
    return ll
rhos=np.linspace(-0.25,0.10,36); lls=[loglik(r) for r in rhos]; RHO=float(rhos[int(np.argmax(lls))])
print(f"Dixon-Coles rho (MLE) = {RHO:.3f}  (neg => draws/low scores more likely than independent Poisson)")

# --- re-validate M3 with vs without DC on the 24 knockouts ---
def val_with_rho(R_,rho):
    P=np.array([advp(R_,TI[r.home_team_id],TI[r.away_team_id],rho=rho) for r in ko.itertuples()]); return metrics(P)
a0=val_with_rho(RM3,0.0); a1=val_with_rho(RM3,RHO)
print(f"M3 no-DC : acc {a0[0]:.3f} logloss {a0[1]:.3f} brier {a0[2]:.3f}")
print(f"M3 +DC   : acc {a1[0]:.3f} logloss {a1[1]:.3f} brier {a1[2]:.3f}")

# --- suspension: red card in R16 -> out for QF ---
sq_pos = sq.set_index('player_id')['posg'].to_dict()
pmin = pstats.set_index('player_id')['minutes_played'].to_dict()
susp = reds[['player_id','team_id','player']].copy()
susp['pos']=susp.player_id.map(sq_pos); susp['mins']=susp.player_id.map(pmin)
print("\nQF suspensions (red in R16):")
print(susp.assign(team=susp.team_id.map(TEAM)).to_string(index=False))
# build per-team QF rating penalty (defender/GK -> worse defense; att -> worse attack), scaled by minutes
QF_PEN={}  # team_id -> (atk_mult, dfn_mult)
for _,s in susp.iterrows():
    share=min(s['mins']/450.0,1.0)  # 450 ~ full-tournament starter minutes
    am,dm=QF_PEN.get(s.team_id,(1.0,1.0))
    if s['pos'] in ('DEF','GK'): dm*= (1+0.18*share)   # concede a bit more
    else: am*= (1-0.22*share)
    QF_PEN[s.team_id]=(am,dm)
print("\nQF penalty multipliers (atk,dfn):", {TEAM[k]:tuple(round(x,3) for x in v) for k,v in QF_PEN.items()})

Dixon-Coles rho (MLE) = -0.230  (neg => draws/low scores more likely than independent Poisson)
M3 no-DC : acc 0.750 logloss 0.543 brier 0.179
M3 +DC   : acc 0.750 logloss 0.541 brier 0.178

QF suspensions (red in R16):
 player_id  team_id                player pos  mins    team
      1170       45 Jarell Amorin Quansah DEF   117 England

QF penalty multipliers (atk,dfn): {'England': (1.0, 1.047)}


## v2.3 · Production model, point estimates & bootstrap uncertainty
Production ratings = recency-weighted, Elo+squad prior, all 96 matches. Analytic advance probability now includes **extra time** before shootouts, plus Dixon-Coles and the QF suspension. Then **block-bootstrap** (resample matches → refit → re-simulate, 300×) for credible intervals.

In [10]:
AL = list(alive.values())               # 8 alive team_ids
ALn = {t:TEAM[t] for t in AL}
def analytic_adv(R, ai, bi, rho=RHO, penpen=None, atk_pen=(1,1), dfn_pen=(1,1)):
    la=R['mu']*R['atk'][ai]*atk_pen[0]*R['dfn'][bi]*dfn_pen[1]
    lb=R['mu']*R['atk'][bi]*atk_pen[1]*R['dfn'][ai]*dfn_pen[0]
    pAr,pdr,pBr=mprobs(la,lb,rho=rho)
    pAe,pde,pBe=mprobs(la/3,lb/3,rho=rho)          # extra time ~ 1/3 match
    pen=1/(1+np.exp(-0.4*(strn(R,ai)-strn(R,bi)))) if penpen is None else penpen
    return pAr + pdr*(pAe + pde*pen)

def padv_matrix(R, qf_pen=None):
    P=np.zeros((len(AL),len(AL)))
    for x,a in enumerate(AL):
        for y,b in enumerate(AL):
            if x==y: continue
            ap=dp=(1,1)
            if qf_pen:
                pa=qf_pen.get(a,(1,1)); pb=qf_pen.get(b,(1,1))
                ap=(pa[0],pb[0]); dp=(pa[1],pb[1])
            P[x,y]=analytic_adv(R,TI[a],TI[b],atk_pen=ap,dfn_pen=dp)
    return P

def simulate(Padv, Pqf, N, rng):
    ix={t:k for k,t in enumerate(AL)}
    def pick(a,b,P):  # vectorized winners over N sims
        pa=P[np.vectorize(ix.get)(a), np.vectorize(ix.get)(b)]
        return np.where(rng.random(N)<pa, a, b)
    F,M,S=alive['France'],alive['Morocco'],alive['Spain'],  # noqa
    a=alive
    w97=pick(np.full(N,a['France']),np.full(N,a['Morocco']),Padv)
    w98=pick(np.full(N,a['Spain']),np.full(N,a['Belgium']),Padv)
    w99=pick(np.full(N,a['Norway']),np.full(N,a['England']),Pqf)     # suspension applies here
    w100=pick(np.full(N,a['Argentina']),np.full(N,a['Switzerland']),Padv)
    sf=np.stack([w97,w98,w99,w100])
    f1=pick(w97,w98,Padv); l1=np.where(f1==w97,w98,w97)
    f2=pick(w99,w100,Padv); l2=np.where(f2==w99,w100,w99)
    champ=pick(f1,f2,Padv); third=pick(l1,l2,Padv)
    fin=np.stack([f1,f2])
    out={}
    for t in AL:
        out[t]=dict(sf=(sf==t).any(0).mean(), final=(fin==t).any(0).mean(),
                    champ=(champ==t).mean(), third=(third==t).mean())
    return out, champ, fin

Rprod=Rprod0
Padv=padv_matrix(Rprod); Pqf=padv_matrix(Rprod, qf_pen=QF_PEN)
rng=np.random.default_rng(11)
pt,champ_pt,fin_pt=simulate(Padv,Pqf,60000,rng)
proj2=pd.DataFrame([{'team':TEAM[t],'reach_SF':pt[t]['sf'],'reach_Final':pt[t]['final'],
                     'champion':pt[t]['champ'],'third':pt[t]['third']} for t in AL]).sort_values('champion',ascending=False)
print("v2 PRODUCTION projection (60k sims, recency+DC+suspension):")
print(proj2.round(3).to_string(index=False))

# QF advance probs (v2) with suspension on Norway-England
qf_v2=[]
for tag,A,B,P in [('QF97','France','Morocco',Padv),('QF98','Spain','Belgium',Padv),
                  ('QF99','Norway','England',Pqf),('QF100','Argentina','Switzerland',Padv)]:
    ix={t:k for k,t in enumerate(AL)}; pa=P[ix[alive[A]],ix[alive[B]]]
    qf_v2.append((tag,A,B,pa)); print(f"  {A} {pa*100:.1f}% vs {B} {(1-pa)*100:.1f}%")

v2 PRODUCTION projection (60k sims, recency+DC+suspension):
       team  reach_SF  reach_Final  champion  third
      Spain     0.688        0.389     0.277  0.215
     France     0.640        0.350     0.252  0.209
  Argentina     0.626        0.344     0.127  0.111
     Norway     0.552        0.286     0.092  0.096
    Morocco     0.360        0.150     0.087  0.123
    England     0.448        0.219     0.068  0.077
    Belgium     0.312        0.110     0.057  0.106
Switzerland     0.374        0.152     0.039  0.063
  France 63.9% vs Morocco 36.1%
  Spain 69.1% vs Belgium 30.9%
  Norway 55.2% vs England 44.8%
  Argentina 62.8% vs Switzerland 37.2%


In [11]:
# precompute arrays for fast bootstrap refits
gsf=(0.6*tm2.xg_for+0.4*tm2.gf).values; gsa=(0.6*tm2.xg_ag+0.4*tm2.ga).values
iarr=tm2.team_id.map(TI).values; jarr=tm2.opp_id.map(TI).values
harr=tm2.is_home.values.astype(float); warr=tm2['w'].values; midarr=tm2.match_id.values
rows_by_match={}
for k,m in enumerate(midarr): rows_by_match.setdefault(m,[]).append(k)
match_ids=comp.match_id.values

def fit_arr(sel, n_iter=140):
    i,j,h=iarr[sel],jarr[sel],harr[sel]; gf_,ga_=gsf[sel],gsa[sel]; w=warr[sel]
    mu=np.average(gf_,weights=w); atk=np.ones(NT); dfn=np.ones(NT); gamma=1.3
    present=np.bincount(i,minlength=NT)>0
    for _ in range(n_iter):
        ha=np.where(h==1,gamma,1.); ho=np.where(h==0,gamma,1.)
        na=np.bincount(i,w*gf_,NT); da=np.bincount(i,w*mu*dfn[j]*ha,NT)
        nd=np.bincount(i,w*ga_,NT); dd=np.bincount(i,w*mu*atk[j]*ho,NT)
        atk=np.where(present&(da>0),na/np.where(da>0,da,1),atk); dfn=np.where(present&(dd>0),nd/np.where(dd>0,dd,1),dfn)
        atk[present]/=atk[present].mean(); dfn[present]/=dfn[present].mean()
        m=h==1; gamma=(w[m]*gf_[m]).sum()/max((w[m]*mu*atk[i[m]]*dfn[j[m]]).sum(),1e-9)
    mp=np.bincount(i,w,NT); sw=mp/(mp+5.0)
    atk=np.exp(sw*np.log(np.clip(atk,1e-6,None))+(1-sw)*( 0.16*np.nan_to_num(priorz_full)))
    dfn=np.exp(sw*np.log(np.clip(dfn,1e-6,None))+(1-sw)*(-0.16*np.nan_to_num(priorz_full)))
    return dict(mu=mu,atk=atk,dfn=dfn,gamma=gamma)

B=300; rngb=np.random.default_rng(99)
boot={t:{'champ':[],'final':[],'sf':[]} for t in AL}
for b in range(B):
    ids=rngb.choice(match_ids,size=len(match_ids),replace=True)
    sel=np.concatenate([rows_by_match[m] for m in ids])
    Rb=fit_arr(sel)
    Pb=padv_matrix(Rb); Pqb=padv_matrix(Rb,qf_pen=QF_PEN)
    r,_,_=simulate(Pb,Pqb,3000,rngb)
    for t in AL:
        boot[t]['champ'].append(r[t]['champ']); boot[t]['final'].append(r[t]['final']); boot[t]['sf'].append(r[t]['sf'])

def ci(t,k): a=np.array(boot[t][k])*100; return np.percentile(a,5),np.percentile(a,95)
ciq=pd.DataFrame([{'team':TEAM[t],
    'champ':round(pt[t]['champ']*100,1),'champ_lo':round(ci(t,'champ')[0],1),'champ_hi':round(ci(t,'champ')[1],1),
    'final':round(pt[t]['final']*100,1),'final_lo':round(ci(t,'final')[0],1),'final_hi':round(ci(t,'final')[1],1)}
    for t in AL]).sort_values('champ',ascending=False)
print("v2 champion odds with 90% bootstrap credible intervals (B=300):")
print(ciq.to_string(index=False))
sp=np.array(boot[alive['Spain']]['champ']); fr=np.array(boot[alive['France']]['champ'])
print(f"\nP(Spain champ-odds > France champ-odds) across bootstraps = {(sp>fr).mean():.2f}  -> favorites are {'separable' if (sp>fr).mean()>0.8 else 'NOT cleanly separable'}")

v2 champion odds with 90% bootstrap credible intervals (B=300):
       team  champ  champ_lo  champ_hi  final  final_lo  final_hi
      Spain   27.7       7.7      43.2   38.9      16.4      55.6
     France   25.2       6.2      42.4   35.0      13.4      54.7
  Argentina   12.7       3.9      24.9   34.4      16.7      53.7
     Norway    9.2       2.6      30.0   28.6      12.3      59.0
    Morocco    8.7       1.8      28.2   15.0       5.0      40.0
    England    6.8       1.0      15.1   21.9       6.8      36.0
    Belgium    5.7       2.3      16.3   11.0       5.7      27.4
Switzerland    3.9       0.4      10.9   15.2       3.7      24.5

P(Spain champ-odds > France champ-odds) across bootstraps = 0.52  -> favorites are NOT cleanly separable


## v2.4 · Bracket-pairing sensitivity
QF98/QF100 are inferred. The 4 unassigned teams {Spain, Belgium, Argentina, Switzerland} can pair 6 admissible ways. The decision-relevant axis: is **Spain in France's half** (they collide in the SF) or the opposite half (both can reach the Final)? Champion odds under every admissible bracket.

In [13]:
a=alive
IDXA=np.full(max(AL)+1,-1); 
for k,t in enumerate(AL): IDXA[t]=k
def gen_sim(qf98, qf100, N, rng):
    def pick(x,y,P):
        x=np.broadcast_to(np.asarray(x),(N,)); y=np.broadcast_to(np.asarray(y),(N,))
        return np.where(rng.random(N)<P[IDXA[x],IDXA[y]], x, y)
    w97=pick(a['France'],a['Morocco'],Padv); w98=pick(qf98[0],qf98[1],Padv)
    w99=pick(a['Norway'],a['England'],Pqf); w100=pick(qf100[0],qf100[1],Padv)
    f1=pick(w97,w98,Padv); f2=pick(w99,w100,Padv); champ=pick(f1,f2,Padv)
    return {t:(champ==a[t2]).mean() for t2,t in [('Spain','Spain'),('France','France'),('Argentina','Argentina'),
            ('Norway','Norway'),('Morocco','Morocco'),('England','England'),('Belgium','Belgium'),('Switzerland','Switzerland')]}
S=lambda n:a[n]
configs={
 'A · ESP-BEL / ARG-SUI  (inferred)': ((S('Spain'),S('Belgium')),(S('Argentina'),S('Switzerland'))),
 'B · ARG-SUI / ESP-BEL':             ((S('Argentina'),S('Switzerland')),(S('Spain'),S('Belgium'))),
 'C · ESP-ARG / BEL-SUI':             ((S('Spain'),S('Argentina')),(S('Belgium'),S('Switzerland'))),
 'D · BEL-SUI / ESP-ARG':             ((S('Belgium'),S('Switzerland')),(S('Spain'),S('Argentina'))),
 'E · ESP-SUI / ARG-BEL':             ((S('Spain'),S('Switzerland')),(S('Argentina'),S('Belgium'))),
 'F · ARG-BEL / ESP-SUI':             ((S('Argentina'),S('Belgium')),(S('Spain'),S('Switzerland'))),
}
rngc=np.random.default_rng(5)
srows=[]
for name,(q98,q100) in configs.items():
    r=gen_sim(q98,q100,40000,rngc)
    esp_top = S('Spain') in q98   # Spain in France's half?
    srows.append({'config':name,'ESP&FRA_same_half':esp_top,
                  'Spain':round(r['Spain']*100,1),'France':round(r['France']*100,1),'Argentina':round(r['Argentina']*100,1)})
sens=pd.DataFrame(srows)
print("Champion odds by admissible bracket pairing:")
print(sens.to_string(index=False))
same=sens[sens['ESP&FRA_same_half']]; opp=sens[~sens['ESP&FRA_same_half']]
print(f"\nSpain champ: same-half {same.Spain.mean():.1f}%  vs  opposite-half {opp.Spain.mean():.1f}%")
print(f"France champ: same-half {same.France.mean():.1f}%  vs  opposite-half {opp.France.mean():.1f}%")
print(f"Spain champ range across brackets: {sens.Spain.min()}–{sens.Spain.max()}%")

Champion odds by admissible bracket pairing:
                           config  ESP&FRA_same_half  Spain  France  Argentina
A · ESP-BEL / ARG-SUI  (inferred)               True   27.2    25.4       12.6
            B · ARG-SUI / ESP-BEL              False   30.3    27.3       10.0
            C · ESP-ARG / BEL-SUI               True   27.0    25.1        6.8
            D · BEL-SUI / ESP-ARG              False   29.3    27.6        7.6
            E · ESP-SUI / ARG-BEL               True   30.1    25.1       10.3
            F · ARG-BEL / ESP-SUI              False   33.1    26.1        8.7

Spain champ: same-half 28.1%  vs  opposite-half 30.9%
France champ: same-half 25.2%  vs  opposite-half 27.0%
Spain champ range across brackets: 27.0–33.1%


## v2.5 · New visuals
Champion odds with credible intervals, the ablation, bracket-pairing sensitivity, per-QF scoreline heatmaps, and a bracket tree.

In [14]:
from plotly.subplots import make_subplots
def savej(fig,name):
    fig.update_layout(font=dict(family="Inter, sans-serif"), margin=dict(l=60,r=30,t=60,b=50))
    d=json.loads(fig.to_json()); d.get('layout',{}).pop('template',None)
    json.dump(d,open(f'{CWD}/{name}.json','w'),separators=(',',':')); return os.path.getsize(f'{CWD}/{name}.json')
V2={}

# FIG A: champion odds with 90% CI
o=ciq.sort_values('champ')
fA=go.Figure()
fA.add_bar(y=o.team,x=o.champ,orientation='h',marker_color=[COL[t] for t in o.team],
   error_x=dict(type='data',symmetric=False,array=o.champ_hi-o.champ,arrayminus=o.champ-o.champ_lo,color='#555',thickness=1.5),
   text=[f"{c:.0f}%" for c in o.champ],textposition='outside',hovertemplate='%{y}: %{x}% [%{customdata[0]}–%{customdata[1]}]<extra></extra>',
   customdata=np.stack([o.champ_lo,o.champ_hi],axis=1))
fA.update_layout(title='Champion odds with 90% credible interval (bootstrap)',xaxis_title='Champion probability (%)',xaxis_range=[0,55])
V2['ci']=savej(fA,'v2_ci')

# FIG B: ablation (log-loss; lower=better)
mo=abl.copy(); cols=['#2E7D32' if 'Elo-only' in m else ('#888' if 'no prior' in m else '#4FA9E0') for m in mo.model]
fB=go.Figure(go.Bar(x=mo.model,y=mo.log_loss,marker_color=cols,text=[f"{l:.3f}" for l in mo.log_loss],textposition='outside'))
fB.add_hline(y=0.693,line=dict(color='#bbb',dash='dash'),annotation_text='coin-flip 0.693')
fB.update_layout(title='Ablation — out-of-sample log-loss on 24 knockouts (lower = better)',yaxis_title='log-loss',yaxis_range=[0.4,0.72])
V2['abl']=savej(fB,'v2_abl')

# FIG C: bracket-pairing sensitivity (Spain/France/Argentina across configs)
fC=go.Figure()
for tm_,c in [('Spain',COL['Spain']),('France',COL['France']),('Argentina',COL['Argentina'])]:
    fC.add_bar(x=sens.config,y=sens[tm_],name=tm_,marker_color=c)
fC.update_layout(barmode='group',title='Champion odds across all 6 admissible bracket pairings',
   yaxis_title='Champion %',xaxis_tickangle=-20,legend=dict(orientation='h',y=-0.35))
V2['sens']=savej(fC,'v2_sens')

# FIG D: scoreline heatmaps for the 4 QFs (DC-corrected, 0..5)
QFdef=[('France','Morocco',(1,1)),('Spain','Belgium',(1,1)),('Norway','England',QF_PEN.get(alive['England'],(1,1))),('Argentina','Switzerland',(1,1))]
def score_mat(A,B,pen,mx=5):
    ai,bi=TI[alive[A]],TI[alive[B]]
    la=Rprod['mu']*Rprod['atk'][ai]*Rprod['dfn'][bi]; lb=Rprod['mu']*Rprod['atk'][bi]*Rprod['dfn'][ai]*pen[1]
    a=poisson.pmf(np.arange(mx+1),la); b=poisson.pmf(np.arange(mx+1),lb); M=np.outer(a,b)
    t=np.ones((mx+1,mx+1)); t[0,0]=1-la*lb*RHO; t[0,1]=1+la*RHO; t[1,0]=1+lb*RHO; t[1,1]=1-RHO
    M=M*t; M/=M.sum(); return M*100
fD=make_subplots(rows=2,cols=2,subplot_titles=[f"{A} (rows) vs {B} (cols)" for A,B,_ in QFdef],
    horizontal_spacing=0.13,vertical_spacing=0.16)
for k,(A,B,pen) in enumerate(QFdef):
    M=score_mat(A,B,pen); r,c=k//2+1,k%2+1
    fD.add_trace(go.Heatmap(z=M,x=[str(i) for i in range(6)],y=[str(i) for i in range(6)],
        colorscale='Blues',showscale=(k==0),zmin=0,zmax=18,hovertemplate=f'{A} %{{y}}–%{{x}} {B}: %{{z:.1f}}%<extra></extra>'),row=r,col=c)
for i in range(1,5):
    fD.layout.annotations[i-1].font.size=11
fD.update_layout(title_text='Quarter-final scoreline probabilities (%) — Dixon-Coles',height=560)
fD.update_xaxes(title_text='away goals',title_font_size=9); fD.update_yaxes(title_text='home goals',title_font_size=9)
V2['heat']=savej(fD,'v2_heat')
print("figure sizes:",V2)

figure sizes: {'ci': 1292, 'abl': 817, 'sens': 1335, 'heat': 5178}


In [15]:
ixm={t:k for k,t in enumerate(AL)}
def qadv(A,B,P=Padv): return P[ixm[alive[A]],ixm[alive[B]]]
# advance % per team in its QF
qf_adv={'France':qadv('France','Morocco'),'Morocco':1-qadv('France','Morocco'),
        'Spain':qadv('Spain','Belgium'),'Belgium':1-qadv('Spain','Belgium'),
        'Norway':qadv('Norway','England',Pqf),'England':1-qadv('Norway','England',Pqf),
        'Argentina':qadv('Argentina','Switzerland'),'Switzerland':1-qadv('Argentina','Switzerland')}
rf={TEAM[t]:pt[t]['final'] for t in AL}; ch={TEAM[t]:pt[t]['champ'] for t in AL}

pairs=[('France','Morocco',9,8),('Spain','Belgium',6,5),('Norway','England',3,2),('Argentina','Switzerland',0,-1)]
fE=go.Figure(); shp=[]
def line(x0,y0,x1,y1): shp.append(dict(type='line',x0=x0,y0=y0,x1=x1,y1=y1,line=dict(color='#c2c8d0',width=1.5)))
railx=[]
for A,B,yh,yl in pairs:
    ym=(yh+yl)/2
    for nm,yy in [(A,yh),(B,yl)]:
        line(0.05,yy,0.9,yy)
        fE.add_scatter(x=[0.05],y=[yy],mode='markers+text',marker=dict(size=15,color=COL[nm],line=dict(color='white',width=1)),
            text=[f"  {nm} · {qf_adv[nm]*100:.0f}%"],textposition='middle right',textfont=dict(size=11),showlegend=False,
            hovertemplate=f"{nm}: advance {qf_adv[nm]*100:.0f}% · reach final {rf[nm]*100:.0f}% · champion {ch[nm]*100:.0f}%<extra></extra>")
    line(0.9,yl,0.9,yh); line(0.9,ym,1.9,ym); railx.append(ym)
# SF verticals
sf1=(railx[0]+railx[1])/2; sf2=(railx[2]+railx[3])/2
line(1.9,railx[1],1.9,railx[0]); line(1.9,sf1,2.9,sf1)
line(1.9,railx[3],1.9,railx[2]); line(1.9,sf2,2.9,sf2)
# Final vertical
line(2.9,sf2,2.9,sf1); line(2.9,(sf1+sf2)/2,3.5,(sf1+sf2)/2)
# labels
fE.add_annotation(x=1.4,y=sf1,text=f"SF101<br>→Final {max(rf['France'],rf['Spain']):.0%} (ESP/FRA)",showarrow=False,font=dict(size=9,color='#666'))
fE.add_annotation(x=1.4,y=sf2,text=f"SF102<br>→Final {rf['Argentina']:.0%} (ARG)",showarrow=False,font=dict(size=9,color='#666'))
fE.add_annotation(x=3.5,y=(sf1+sf2)/2+1.1,text="🏆 Champion",showarrow=False,font=dict(size=12,color='#222'))
fE.add_annotation(x=3.5,y=(sf1+sf2)/2,text=f"Spain 28%<br>France 25%<br>Argentina 13%",showarrow=False,font=dict(size=10,color='#222'),align='left')
for xx,lab in [(0.05,'QUARTER-FINAL'),(1.9,'SEMI-FINAL'),(2.9,'FINAL')]:
    fE.add_annotation(x=xx,y=10.2,text=lab,showarrow=False,font=dict(size=10,color='#999'),xanchor='left')
fE.update_layout(title='WC26 bracket — advance % (labels), reach-final % (hover)',shapes=shp,height=520,
    xaxis=dict(visible=False,range=[-0.1,4.2]),yaxis=dict(visible=False,range=[-1.8,10.6]),plot_bgcolor='white')
V2['tree']=savej(fE,'v2_tree')
print("bracket tree bytes:",V2['tree'])
print("all v2 figs:",list(V2))

bracket tree bytes: 5762
all v2 figs: ['ci', 'abl', 'sens', 'heat', 'tree']
